In [39]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

ENGINE_ROLE = {"role": "system", "content": (
                "You are an expert analyst. Process the search results according to these rules:\n"
                "1. Prioritize results by weighting Accuracy (50%) and Recency (50%).\n"
                "2. Provide the top 5 findings concisely.\n"
                "3. ALWAYS respond in English, no matter what language the user uses.\n"
                "4. Base your answer strictly on the provided search context from the last year."
            )}

In [44]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [45]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)


{'result': [{'url': 'https://media.jreast.co.jp/tags/9756',
   'title': '東京駅イベントに関する最新情報・おすすめ記事 | JREメディア',
   'content': ':   「東京駅イベント」でタグ付けされた記事一覧です。JREメディアには「東京駅イベント」に関する記事やご案内、便利な情報が8件掲載されています。. # 東京駅で群馬溫泉文化旅遊！温泉王国ぐんま体感イベントを開催. 2026年2月20日（金）～22日（日）に、JR東日本東京駅B1「スクエア ゼロ」で開催される「群馬溫泉... # 東京駅で開催！青森・北海道道南産直市＠スクエアゼロ. 「青森県・函館観光キャンペーン」期間中、JR東日本クロスステーションは、エキナカ地域フェア「青森・北海道... # 東京駅で開催中の春イベント「TOKYO EASTER & SWEETS」をレポート！. 2025年4月20日(日)まで東京駅特設会場で開催中の「TOKYO EASTER&SWEETS」。対象店... # 東京駅発春イベント「TOKYO EASTER & SWEETS」開催！. JR東京駅B1改札内イベントスペース「スクエア ゼロ」にて、春の訪れを祝福するお祭り”イースター”をテー... # クリスマスは謎解きを楽しもう♪『東京駅サンタ謎～110年目のプレゼント～』開催！. # 東京駅開業110周年記念イベント12月開催決定！そのイベントの内容とは!? 東京駅は2024年12月に開業110周年を迎えます。東京駅は、東北・上越・北陸新幹線や東海道新幹線の発着... # 東京駅で北陸フェア開催！北陸の”おいしい”が大集合【のもの東京】. 北陸デスティネーションキャンペーンに合わせて、2024年11月19日(火)～2024年12月2日(月)ま... # 東京駅で長野フェア開催！長野の“おいしい”が大集合【のもの東京】. “ココロうごく体験”をお届けする夏の信州観光キャンペーンに合わせて、2024年9月17日(火)～2024... ## アクセスランキング. ### 平日限定で50％割引！新幹線や特急列車に乗っておトクに平日旅♪. ### 東京駅でしか買えない限定お土産25選【2026年最新版】. ### みどりの窓口のある駅・営業時間【2

In [46]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": ENGINE_ROLE.get("content"),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

In [47]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            ENGINE_ROLE,
            {"role": "user", "content": question}
        ],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [48]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            ENGINE_ROLE,
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call


In [49]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [ ]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。

- **東京都**の面積は約2,194平方キロメートルです。
- **沖縄県**の面積は約2,271平方キロメートルです。

したがって、沖縄県の方が東京都よりも面積が広いです。


In [ ]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近の東京駅に関するイベント情報を以下にまとめました。

1. **東京駅周辺のイベント一覧**  
   - **URL**: [東京都周辺イベント](https://bestcalendar.jp/events/%E6%9D%B1%E4%BA%AC%E9%A7%85)  
   - 2024年4月のイベント情報が掲載されています。東京駅周辺の最新のイベントや発表日の情報をチェックできます。

2. **東京駅 慶應のイベント**  
   - **URL**: [東急線 東京駅イベント](https://www.walkerplus.com/event_list/ar0313/sc309880d/)  
   - 東急線の駅で行われるイベント情報があり、特に祭りや特別な展示なども含まれています。

3. **東京駅サイファー関連のポップアップショップ**  
   - **URL**: [東京駅一番街イベント情報](https://www.tokyoeki-1bangai.co.jp/event/)  
   - テレビアニメ「夏目友人帳」のポップアップショップなど、様々なイベントが行われています。

4. **東京のイベント情報全般**  
   - **URL**: [TOKYO FES 2026](https://www.tokyofes.info/)  
   - 東京全体のイベント情報が網羅されており、特にフリーマーケットや文化イベントなどがあります。

5. **新しい展示や特集展示について**  
   - **URL**: [ENJOY TOKYO](https://www.enjoytokyo.jp/event/list/area1306/)  
   - 2026年の新しい展示や特集が計画されており、楽しむための多くの情報がまとめられています。

最新の情報や詳細は、各リンクを参照してください。東京駅周辺での楽しいイベントがいくつも計画されていますので、是非参加してみてください。


In [50]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 3:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:ၵႂၢမ်းလၢတ်ႈ ဢၼ်ၵူၼ်းၼုမ်ႇ ၸၢဝ်းသပႃးၼိတ်ႉၶဝ် ၸႂ်ႉတိုဝ်း'

Based on the recent findings, here are the top 5 results regarding Myanmar's politics and economy:

1. **[The Economist - Myanmar](https://www.economist.com/topics/myanmar?after=0e5817f7-dd02-4730-bfca-1d5bf15bfdc6)**: Offers comprehensive coverage on Myanmar's political landscape, economic developments, and cultural insights. Recent articles discuss political changes and the junta's position.

2. **[Frontier Myanmar](https://www.frontiermyanmar.net/en/)**: Reports that Myanmar's coup leader, Senior General Min Aung Hlaing, is set to retain power as president. It highlights the ongoing political maneuvering surrounding governance in the country.

3. **[Asian Development Bank - Myanmar's Economy](https://www.adb.org/where-we-work/myanmar/economy)**: Provides economic forecasts, noting expected GDP growth of 2.4% in 2026 and 2.7% in 2027, indicating slow recovery post-coup.

4. **[Wall Street Journal - Myanmar Updates](https://www.wsj.com/topics/place/myanmar?eafs_enabled=false)**: Descr